In [ ]:
from transformers import AutoTokenizer, AutoModelForMultipleChoice
from peft import LoraConfig, get_peft_model

MODEL_NAME = "microsoft/deberta-v3-base"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(example):
    first  = [example["prompt_clean"]] * 5      # question, repeated 5 times
    second = [example[o] for o in OPTS]         # the five options
    tok = tokenizer(first, second, truncation=True, max_length=256)
    out = {k: v for k, v in tok.items()}
    if "label" in example:
        out["label"] = example["label"]
    return out

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.1,
    target_modules=["query_proj", "key_proj", "value_proj"],   # DeBERTa naming
    modules_to_save=["classifier", "pooler"],                  # new head, train fully
    task_type="SEQ_CLS",
)

deberta = get_peft_model(
    AutoModelForMultipleChoice.from_pretrained(MODEL_NAME), lora_config)
deberta.print_trainable_parameters()
# trainable params: 1,476,097 || all params: 185,899,010 || trainable%: 0.7940